# 分支限界法

## 背包问题

分支限界法是一种系统地搜索解空间的方法，适用于解决组合优化问题，如背包问题。背包问题的目标是在给定容量的背包中选择物品，使得总价值最大化。

思路:

1. **分支**：将问题分解成子问题。例如，在背包问题中，可以选择放入当前物品或不放入当前物品。
2. **限界**：计算当前分支的上界（即在当前选择的基础上，能获得的最大价值）。如果上界小于当前已知的最优解，则可以剪枝，避免继续搜索该分支。
以下是使用分支限界法解决背包问题的示例代码：


In [ ]:
import heapq

def knapsack_bound(weights, values, capacity):
    n = len(weights)

    # 按价值密度倒序排序
    items = sorted(
        [(values[i] / weights[i], weights[i], values[i], i) for i in range(n)],
        reverse=True
    )

    # 计算上界
    def bound(level, current_weight, current_value):
        # level: 当前考虑的物品索引
        # current_weight: 当前已选物品的总重量
        # current_value: 当前已选物品的总价值

        if current_weight >= capacity: # 如果当前重量已经超过容量，上界设为 0
            return 0
        total_value = current_value
        total_weight = current_weight

        # 估计上界 # 按物品可分割的情况按价值密度顺序尽量装入
        for k in range(level, n):
            ratio, w, v, _ = items[k]
            if total_weight + w <= capacity:
                total_weight += w
                total_value += v
            else:
                total_value += (capacity - total_weight) * ratio
                break
        return total_value

    best_value = 0
    best_taken = [0] * n

    # max-heap by negative bound
    heap = [(-bound(0, 0, 0), 0, 0, 0, [])] # 节点中储存: 上界, 物品id, 已载重量, 已载物品价值, 已选物品列表
    while heap:
        # 取出顶部节点
        _, level, current_weight, current_value, taken = heapq.heappop(heap)

        if level == n:
            if current_value > best_value:
                best_value = current_value
                best_taken = taken
            continue

        ratio, w, v, idx = items[level]

        # take the item
        new_weight = current_weight + w
        new_value = current_value + v
        new_taken = taken + [idx]

        if new_weight <= capacity and new_value > best_value:
            best_value = new_value
            best_taken = new_taken

        # 
        if new_weight <= capacity:
            b = bound(level + 1, new_weight, new_value)
            if b > best_value:
                heapq.heappush(heap, (-b, level + 1, new_weight, new_value, new_taken))

        # 
        b = bound(level + 1, current_weight, current_value)
        if b > best_value:
            heapq.heappush(heap, (-b, level + 1, current_weight, current_value, taken))

    selected = [0] * n
    for idx in best_taken:
        selected[idx] = 1

    return best_value, selected


# 示例
weights = [2, 2, 6, 5, 4]
values = [6, 3, 5, 4, 6]
capacity = 10

best_value, selected = knapsack_bound(weights, values, capacity)
print("最优价值:", best_value)
print("选择情况:", selected)

## TSP 问题

旅行商问题（Traveling Salesman Problem，TSP）。

分支限界法可以用于解决 TSP 问题。
```mermaid
graph TD
    A[分支限界法] --> B[TSP 问题]
    B --> C[分支：选择下一个城市]
    B --> D[限界：计算上界，剪枝]
```



In [ ]:
def tsp_branch_and_bound(dist, start=0):
    n = len(dist)
    best_cost = float("inf")
    best_path = None

    def mst_cost(nodes):
        nodes = list(nodes)
        if len(nodes) <= 1:
            return 0
        in_tree = {nodes[0]}
        remaining = set(nodes[1:])
        total = 0
        while remaining:
            w, nxt = min(
                (dist[i][j], j)
                for i in in_tree
                for j in remaining
            )
            total += w
            in_tree.add(nxt)
            remaining.remove(nxt)
        return total

    def lower_bound(path, visited, cost):
        last = path[-1]
        unvisited = set(range(n)) - visited
        if not unvisited:
            return cost + dist[last][start]
        min_last = min(dist[last][u] for u in unvisited)
        min_start = min(dist[start][u] for u in unvisited)
        return cost + mst_cost(unvisited) + min_last + min_start

    counter = 0
    heap = [(lower_bound([start], {start}, 0), counter, 0, [start], {start})]

    while heap:
        bound, _, cost, path, visited = heapq.heappop(heap)

        if bound >= best_cost:
            continue

        if len(path) == n:
            tour_cost = cost + dist[path[-1]][start]
            if tour_cost < best_cost:
                best_cost = tour_cost
                best_path = path + [start]
            continue

        last = path[-1]
        for city in range(n):
            if city in visited:
                continue
            new_cost = cost + dist[last][city]
            new_visited = visited | {city}
            new_path = path + [city]
            new_bound = lower_bound(new_path, new_visited, new_cost)
            if new_bound < best_cost:
                counter += 1
                heapq.heappush(heap, (new_bound, counter, new_cost, new_path, new_visited))

    return best_cost, best_path


# 示例
dist = [
    [0, 10, 15, 20],
    [10, 0, 35, 25],
    [15, 35, 0, 30],
    [20, 25, 30, 0]
]

best_cost, best_path = tsp_branch_and_bound(dist)
print("最短路径长度:", best_cost)
print("最优路径:", best_path)